<h2 align="center"> Vehicle Damage Detection Project: Hyperparameter Tunning</h2>

In [1]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import time
import torchvision.models as models
from matplotlib import pyplot as plt
import optuna

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [ ]:
### Load Data

In [3]:
image_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [6]:
dataset_path = r"C:\Users\91994\Downloads\Resources_ModelTraining_CNN\Download files\dataset"

dataset = datasets.ImageFolder(root=dataset_path, transform=image_transforms)
len(dataset)

2300

In [7]:
class_names = dataset.classes
class_names 

['F_Breakage', 'F_Crushed', 'F_Normal', 'R_Breakage', 'R_Crushed', 'R_Normal']

In [8]:
num_classes = len(dataset.classes)
num_classes

6

In [9]:
train_size = int(0.75*len(dataset))
val_size = len(dataset) - train_size

train_size, val_size


(1725, 575)

In [10]:
from torch.utils.data import random_split

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [11]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)

### Model Training & Hyperparameter Tunning

In [16]:
# Load the pre-trained ResNet model
class CarClassifierResNet(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.5):
        super().__init__()
        self.model = models.resnet50(weights= None)
        # Freeze all layers except the final fully connected layer
        for param in self.model.parameters():
            param.requires_grad = False
            
        # Unfreeze layer4 and fc layers
        for param in self.model.layer4.parameters():
            param.requires_grad = True            
            
        # Replace the final fully connected layer
        self.model.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(self.model.fc.in_features, num_classes)
        )

    def forward(self, x):
        x = self.model(x)
        return x

In [17]:
# Define the objective function for Optuna
def objective(trial):
    # Suggest values for the hyperparameters
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.7)
    
    # Load the model
    model = CarClassifierResNet(num_classes=num_classes, dropout_rate=dropout_rate).to(device)
    
    # Define the loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    
    # Training loop (using fewer epochs for faster hyperparameter tuning)
    epochs = 3
    start = time.time()
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_num, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        
        # Validation loop
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        
        # Report intermediate result to Optuna
        trial.report(accuracy, epoch)
        
        # Handle pruning (if applicable)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    end = time.time()
    print(f"Execution time: {end - start} seconds")
    
    return accuracy

In [ ]:
# Create the study and optimize
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

[I 2025-12-29 10:55:58,817] A new study created in memory with name: no-name-411876ec-5298-4569-bc41-bc9f7a527328
[I 2025-12-29 12:15:04,339] Trial 1 finished with value: 31.304347826086957 and parameters: {'lr': 0.0001971194551784367, 'dropout_rate': 0.48222941896064064}. Best is trial 1 with value: 31.304347826086957.


Execution time: 2357.948571920395 seconds


[I 2025-12-29 13:05:08,500] Trial 2 finished with value: 25.217391304347824 and parameters: {'lr': 2.2779151510104545e-05, 'dropout_rate': 0.27564007103062815}. Best is trial 1 with value: 31.304347826086957.


Execution time: 3002.9298901557922 seconds


[I 2025-12-29 13:42:30,145] Trial 3 finished with value: 24.695652173913043 and parameters: {'lr': 0.0012156647510277846, 'dropout_rate': 0.3268126217328524}. Best is trial 1 with value: 31.304347826086957.


Execution time: 2240.0420546531677 seconds


[I 2025-12-29 14:22:51,464] Trial 4 finished with value: 29.043478260869566 and parameters: {'lr': 0.00012635515638076525, 'dropout_rate': 0.3860569992574485}. Best is trial 1 with value: 31.304347826086957.


Execution time: 2419.9934821128845 seconds


[I 2025-12-29 14:36:59,101] Trial 5 pruned. 
[I 2025-12-29 14:50:58,722] Trial 6 pruned. 
[I 2025-12-29 15:04:46,547] Trial 7 pruned. 


In [ ]:
study.best_params